# Weather_GNN: Minimal Training & Inference Demo
This notebook demonstrates a minimal workflow using Weather_GNN for weather prediction.

In [ ]:
# Install dependencies (uncomment for Colab/first run)
# !pip install torch torch-geometric meteostat geopy scikit-learn numpy pandas matplotlib joblib

In [2]:
# Imports (assumes all repo .py files are in the same directory as this notebook)
from Data import Get_UK_Stations, Gather_Data, Connect_Graph, Build_Graph, Normalise_fit_on_train
from model import Weather_GNN
from utils import train, eval, predict
from torch_geometric.loader import DataLoader
import torch, joblib

ModuleNotFoundError: No module named 'Data'

In [1]:
# 1. Prepare data (small subset for demo)
from datetime import datetime
features = ['temp','pres']
start, end = datetime(2022,1,1), datetime(2022,2,1)  # 1 month only
stations_df = Get_UK_Stations().head(10)  # Use only 10 stations for speed
Weather_data = Gather_Data(stations_df, start, end, features)
nHr = len(next(iter(Weather_data.values())))
lookback, pred_window, trainFrac = 24, 6, 0.8
npos = nHr - (pred_window + lookback)
ntrain = int(npos * trainFrac)
last_train_T = lookback + ntrain - 1
train_end_idx = last_train_T + pred_window
Weather_data, scalers = Normalise_fit_on_train(Weather_data, features, train_end_idx)
SIDs = list(Weather_data.keys())
edge_index, edge_attr = Connect_Graph(stations_df, SIDs, nearest=3)
Graph = Build_Graph(Weather_data, edge_index, edge_attr, pred_window, lookback)
train_graphs = Graph[:ntrain]
val_graphs = Graph[ntrain:]


NameError: name 'Get_UK_Stations' is not defined

In [ ]:
# 2. Training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_feat = len(features)
time_step = lookback + 1
model = Weather_GNN(n_feat, 32, n_feat*pred_window, time_step).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
loss_fn = torch.nn.MSELoss()
train_loader = DataLoader(train_graphs, batch_size=2, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=2)
epochs = 3  # Keep it small for demo

for epoch in range(epochs):
    train_loss = train(train_loader, model, optimizer, loss_fn, n_feat, pred_window, device)
    r2, val_loss = eval(val_loader, model, n_feat, pred_window, loss_fn, device)
    print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}, R2 = {r2:.2f}")

In [ ]:
# 3. Inference on a sample graph
test_graph = val_graphs[0].to(device)
preds, targs = predict(test_graph, model, n_feat, pred_window, device)
print("Predictions (first station, first feature):", preds[0][:,0])
print("Targets      (first station, first feature):", targs[0][:,0])